# ¿Añadir el historial de SSN reduce el error?
Se repite el ajuste en las mismas fechas. Delta es error con SSN menos error sin SSN. Un valor negativo indica mejora. Se usan versiones históricas finales de los índices.
Ejecuta las celdas en orden. Las tablas CSV y figuras se guardan en la carpeta de trabajo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# El origen t usa índices hasta t-1 por la ventana centrada trimestral.
# Supuesto retrospectivo: el índice de t-1 está disponible al cierre de t.
def experimento(incluir_solar):
    filas = []
    pronosticos = []
    for v in ["ICEN","ONI"]:
        for h in [3,6,12,24]:
            entradas = {f"{v}_{i}":df[v].shift(i) for i in range(1,13)}
            if incluir_solar:
                entradas.update({f"SSN_{i}":df.SSN.shift(i) for i in range(1,13)})
            X = pd.DataFrame(entradas)
            y = df[v].shift(-h)
            destino = df.index + pd.DateOffset(months=h)
            valido = X.notna().all(axis=1) & y.notna()
            train = valido & (destino < "1995-01-01")
            val = valido & (df.index >= "1995-01-01") & (destino < "2010-01-01")
            test = valido & (df.index >= "2010-01-01")
            errores = []
            for alfa in [0.1,1,10,100]:
                modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
                modelo.fit(X[train],y[train])
                errores.append(np.mean((modelo.predict(X[val])-y[val])**2))
            alfa = [0.1,1,10,100][np.argmin(errores)]
            modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
            ajuste = valido & (destino < "2010-01-01")
            modelo.fit(X[ajuste],y[ajuste])
            modelos = {"Ridge con SSN" if incluir_solar else "Ridge sin SSN":modelo.predict(X[test])}
            if not incluir_solar:
                modelos["Persistencia"] = X.loc[test,f"{v}_1"].to_numpy()
            for etiqueta, pred in modelos.items():
                error = pred-y[test].to_numpy()
                filas.append([v,h,etiqueta,len(pred),np.abs(error).mean(),np.sqrt((error**2).mean()),alfa if etiqueta!="Persistencia" else None])
                pronosticos.extend(zip([v]*len(pred),[h]*len(pred),[etiqueta]*len(pred),df.index[test],destino[test],y[test],pred))
    return (pd.DataFrame(filas,columns=["indice","h","modelo","n","MAE","RMSE","alfa"]),
            pd.DataFrame(pronosticos,columns=["indice","h","modelo","origen","destino","observado","estimado"]))

base_m, _ = experimento(False)
solar, series = experimento(True)
series.to_csv("15_pronosticos_solares.csv",index=False)
metricas = pd.concat([base_m,solar],ignore_index=True)
metricas.to_csv("15_metricas.csv",index=False)
base_r = base_m.query("modelo=='Ridge sin SSN'").set_index(["indice","h"])
solar_r = solar.set_index(["indice","h"])
delta = solar_r[["MAE","RMSE"]]-base_r[["MAE","RMSE"]]
delta.to_csv("15_cambio_error.csv")
print(delta.round(4).to_string())

              MAE    RMSE
indice h                 
ICEN   3  -0.0165 -0.0176
       6  -0.0301 -0.0274
       12 -0.0390 -0.0280
       24 -0.0192 -0.0114
ONI    3   0.0016 -0.0033
       6  -0.0067 -0.0112
       12 -0.0223 -0.0089
       24 -0.0024  0.0069
